### Imports

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

### Reading the data

In [2]:
df = pd.read_csv("./train.csv")

In [3]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df.shape

(42000, 785)

Note: df.shape excludes the header row.<br><br>
Each row is an image sample, so we have 42000 image samples.

In [5]:
# convert to numpy array
df = np.array(df)

# m = number of image samples
# n = number of features + 1 for the 'label' column
m, n = df.shape

### Splitting into validation and training sets

In [6]:
np.random.shuffle(df)

# use 1000 image samples as our validation set
# take the transpose, so now each column is one image sample
validation = df[0:1000].T
print('validation:', validation.shape)

# since we took the transpose, the first row now contains the labels for the 1000 image samples
y_validation = validation[0]
# shape is (1000,) since it's a vector
print('y_validation -> the labels:', y_validation.shape)

# pixel values of all 1000 image samples
x_validation = validation[1:n]
print('x_validation -> the data:', x_validation.shape)

# use the rest 41000 images as our training set
train = df[1000:m].T
print ('\ntrain:', train.shape)

y_train = train[0]
print('y_train:', y_train.shape)

x_train = train[1:n]
print('x_train:', x_train.shape)

validation: (785, 1000)
y_validation -> the labels: (1000,)
x_validation -> the data: (784, 1000)

train: (785, 41000)
y_train: (41000,)
x_train: (784, 41000)


In [7]:
# take all rows but only column 1
# confirm there are 784 rows
x_train[:, 0].shape

(784,)

### Initializing weight matrices and bias vectors

In [8]:
def init_parameters():
    # for W1, create a 10 x 784 matrix
    # row i contains the weights of the 784 connections from our 0th (input) layer to the ith node in our 1st (hidden) layer
    W1 = np.random.randn(10, 784)
    
    # 1 bias value for each of the 10 nodes in our 1st (hidden) layer
    b1 = np.random.randn(10, 1)

    # create a 10 x 10 matrix
    # row i contains the weights of the 10 connections from our 1st (hidden) layer to the ith node in our 2nd (output) layer
    W2 = np.random.randn(10, 10)

    # 1 bias value for each of the 10 nodes in our 2nd (output) layer
    b2 = np.random.randn(10, 1)
  
    return W1, b1, W2, b2

### Defining ReLU function

In [9]:
def relu(Z):
    # if x in Z <= 0 then x = 0; else x = x
    return np.maximum(0, Z)

### Defining softmax function

In [10]:
def softmax(Z):
    # recall softmax takes a vector and converts each value into a probability
    # for each element x in the vector, it calcs [e^x / the sum of e^x over all x's in the vector]
    return np.exp(Z) / np.sum(np.exp(Z))

### Forward propagation (computing $A^{[1]}$ and $A^{[2]}$)

In [11]:
def forward_propagation(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = relu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    
    return Z1, A1, Z2, A2

### Defining one-hot encoding function

In [12]:
def onehot(Y):
    # classes 0-9
    onehotY = np.zeros((Y.size, 10))

    for i in range(Y.size):
        label = Y[i]
        onehotY[i][label] = 1

    # want each column as an example
    onehotY = onehotY.T
    return onehotY

### Defining ReLU derivative function

In [13]:
def relu_derivative(Z):
    # > 0 = slope 1 = true
    return Z > 0

### Backpropagation (computing gradients)

In [14]:
def backpropagation(Z1, A1, A2, W2, X, Y):
    onehotY = onehot(Y)

    dZ2 = A2 - onehotY
    dW2 = 1 / Y.size * dZ2.dot(A1.T)
    db2 = 1 / Y.size * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = W2.T.dot(dZ2) * relu_derivative(Z1)
    dW1 = 1 / Y.size * dZ1.dot(X.T)
    db1 = 1 / Y.size * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

### Updating weights and biases with computed gradients

In [15]:
def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2

    return W1, b1, W2, b2